# S2L replay, final run: Hugging Face -> stages 1-2 -> Hugging Face -> stages 3-4 -> results and plots

Qwen2.5-0.5B + LoRA, task order **GSM8K -> WikiSQL -> XSum -> PubMedQA**, **S2L replay with both buffer sizes (2% and 5%)**, made earlier by the 4 machine notebooks. Each ratio is its own run (`s2l_0.02_s0`, `s2l_0.05_s0`) and they are compared on score, held-out loss, time to converge and peak GPU memory.

| Part | What it does |
|---|---|
| **A** | Pulls `data/` and the S2L `buffers/` from your exchange repo (HF **dataset** repo), trains **stages 1-2** for each ratio, pushes the hand-off (adapter, results, buffers, data, checksums) to a **new folder `s2l_handoff/` in the same HF repo** |
| **B** | Pulls each hand-off, verifies it, trains **stages 3-4**, pushes the final adapters + merged `results.json` |
| **C** | Pulls the baseline runs (`none`, `random`, `full`) from the exchange repo, aggregates everything and compares the two ratios (tables and plots) |

Parts A and B can run in the **same** session or in **different** sessions/accounts: Part B only needs the HF repo and folder. Run sections 1-3 + the setup cell in each session.

**Before you start**
1. Runtime -> Change runtime type -> **GPU**.
2. Add a Hugging Face **write** token in Colab **Secrets** as `HF_TOKEN` (notebook access ON).
3. Have the exchange repo name from the 4 machine notebooks (`S2L_REPO`). The hand-off goes into the same repo under `HF_FOLDER` (default `s2l_handoff/`). If Part B is run by someone else, add them as a collaborator on that repo (it is private).
4. After the `pip install` cell, **restart the session** and continue from section 3.
**Time:** two ratios means two full runs. To split them (two sessions or two accounts), set `RATIOS = [0.02]` in one and `RATIOS = [0.05]` in the other; the runs never overwrite each other. Part C needs both finished.

In [ ]:
!nvidia-smi
import torch; print('cuda:', torch.cuda.is_available(), '| bf16:', torch.cuda.is_available() and torch.cuda.is_bf16_supported())

## 1. Mount Drive and enter the project folder

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/s2l_final'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print('working in', os.getcwd())

## 2. Write the project files (overwrites files with the same name in the folder)
`common.py` is identical to the one used by the 4 machine notebooks, so the buffer indices match the cached data.

In [ ]:
%%writefile common.py
"""
Shared pieces used by every script: config, data loading, prompt format, encoding, metrics.

Tasks (in training order): GSM8K -> WikiSQL -> XSum -> PubMedQA
Target model: Qwen2.5-0.5B (LoRA).  Proxy model for S2L: Pythia-160M.

IMPORTANT: the first call to load_task() caches the selected rows to data/<task>.json.
Buffer files store INDICES into those cached rows, so every teammate must use the same
data/ folder (commit it to git) or the indices will point at the wrong examples.
"""
import json
import os
import re

TARGET_MODEL = "Qwen/Qwen2.5-0.5B"
PROXY_MODEL = "EleutherAI/pythia-160m"
SEED = 0                    # fixes the data subset; run seeds only change training randomness
DATA_DIR = "data"
ORDER = ["gsm8k", "wikisql", "xsum", "pubmedqa"]
MAX_CHARS = 1500            # truncate long inputs (XSum articles, PubMedQA abstracts)

TASKS = {
    "gsm8k":    dict(n_train=7000, n_test=300, max_new=256),   # full GSM8K train is ~7.47k
    "wikisql":  dict(n_train=8000, n_test=300, max_new=96),
    "xsum":     dict(n_train=8000, n_test=300, max_new=64),
    # PubMedQA's labelled set has only 1,000 examples (yes/no/maybe): split 700 train / 300 test
    "pubmedqa": dict(n_train=700,  n_test=300, max_new=6),
}


def root(debug):
    """Debug runs write to ./debug so they never overwrite real results or buffers."""
    return "debug" if debug else "."


# ------------------------------------------------------------------ formatting
def _fmt_gsm8k(x):
    return ("Solve the math problem step by step. End with '#### <answer>'.\n\n" + x["question"],
            x["answer"])


def _fmt_wikisql(x):
    t = x["table"]
    if isinstance(t, str):
        t = json.loads(t)
    s = x["sql"]
    if isinstance(s, str):
        s = json.loads(s)
    cols = " | ".join(t["header"])
    return (f"Write a SQL query that answers the question.\n\nTable columns: {cols}\n\n"
            f"Question: {x['question']}", s["human_readable"])


def _fmt_xsum(x):
    return ("Summarize the article in one sentence.\n\n" + x["document"][:MAX_CHARS],
            x["summary"].strip())


def _fmt_pubmedqa(x):
    ctx = x["context"]
    if isinstance(ctx, dict):
        ctx = ctx["contexts"]
    abstract = " ".join(ctx)[:MAX_CHARS]
    return (f"Answer the question with yes, no, or maybe, using the abstract.\n\n"
            f"Abstract: {abstract}\n\nQuestion: {x['question']}", x["final_decision"])


FMT = {"gsm8k": _fmt_gsm8k, "wikisql": _fmt_wikisql, "xsum": _fmt_xsum, "pubmedqa": _fmt_pubmedqa}


# ------------------------------------------------------------------ loading
def _load_wikisql():
    from datasets import load_dataset
    errors = []
    # WikiSQL on the Hub uses a loading script. That needs datasets<4 and trust_remote_code.
    try:
        return load_dataset("Salesforce/wikisql", trust_remote_code=True)
    except Exception as e:  # noqa: BLE001
        errors.append(f"script load: {e}")
    # Fallback: the auto-converted parquet branch (not verified for this dataset's schema)
    try:
        return load_dataset("Salesforce/wikisql", revision="refs/convert/parquet")
    except Exception as e:  # noqa: BLE001
        errors.append(f"parquet branch: {e}")
    raise RuntimeError(
        "Could not load Salesforce/wikisql. Try: pip install 'datasets>=2.19,<4.0' "
        "(script-based datasets are not supported in datasets>=4).\n" + "\n".join(errors))


def _load_raw(name):
    from datasets import load_dataset
    if name == "gsm8k":
        ds = load_dataset("openai/gsm8k", "main")
        return ds["train"], ds["test"]
    if name == "wikisql":
        ds = _load_wikisql()
        return ds["train"], ds["test"]
    if name == "xsum":
        ds = load_dataset("EdinburghNLP/xsum")
        return ds["train"], ds["test"]
    if name == "pubmedqa":
        ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled")["train"]  # 1,000 rows, one split
        sp = ds.train_test_split(test_size=TASKS[name]["n_test"], seed=SEED)
        return sp["train"], sp["test"]
    raise ValueError(name)


def load_task(name, debug=False):
    """Returns (train_rows, test_rows); each row is {"prompt": str, "response": str}."""
    path = os.path.join(DATA_DIR, f"{name}.json")
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
    else:
        cfg = TASKS[name]
        tr, te = _load_raw(name)
        tr = tr.shuffle(seed=SEED).select(range(min(cfg["n_train"], len(tr))))
        te = te.shuffle(seed=SEED).select(range(min(cfg["n_test"], len(te))))

        def to_rows(ds):
            return [dict(zip(("prompt", "response"), FMT[name](x))) for x in ds]

        d = {"train": to_rows(tr), "test": to_rows(te)}
        os.makedirs(DATA_DIR, exist_ok=True)
        with open(path, "w") as f:
            json.dump(d, f)
    if debug:  # tiny slice for pipeline smoke tests
        d = {"train": d["train"][:50], "test": d["test"][:20]}
    return d["train"], d["test"]


# ------------------------------------------------------------------ prompt + encoding
def build_prompt(p):
    return f"### Instruction:\n{p}\n\n### Response:\n"


def encode(tok, row, maxlen):
    """Token ids + labels. Loss is computed on the response only (prompt tokens are -100)."""
    p = tok(build_prompt(row["prompt"]), add_special_tokens=False).input_ids
    r = tok(row["response"], add_special_tokens=False).input_ids + [tok.eos_token_id]
    return {"input_ids": (p + r)[:maxlen], "labels": ([-100] * len(p) + r)[:maxlen]}


def collate(batch, pad_id):
    import torch
    m = max(len(x["input_ids"]) for x in batch)
    ids = torch.tensor([x["input_ids"] + [pad_id] * (m - len(x["input_ids"])) for x in batch])
    lab = torch.tensor([x["labels"] + [-100] * (m - len(x["labels"])) for x in batch])
    att = torch.tensor([[1] * len(x["input_ids"]) + [0] * (m - len(x["input_ids"])) for x in batch])
    return {"input_ids": ids, "labels": lab, "attention_mask": att}


# ------------------------------------------------------------------ metrics
def clean(pred):
    # Cut off if the model starts a new "### Instruction:" block. Must not match the "####"
    # answer marker used by GSM8K, so require exactly three '#' followed by whitespace.
    return re.split(r"\n###\s", "\n" + pred.strip())[0].strip()


def gsm_ans(s):
    m = re.findall(r"####\s*\$?\s*(-?[\d,]*\.?\d+)", s)
    if not m:
        return None
    try:
        return float(m[-1].replace(",", ""))
    except ValueError:
        return None


def _norm_sql(s):
    return re.sub(r"\s+", " ", s.lower().replace('"', "'").strip().rstrip(";")).strip()


def score(task, preds, refs):
    """gsm8k/wikisql/pubmedqa -> accuracy; xsum -> ROUGE-L F1. All in [0, 1]."""
    preds = [clean(p) for p in preds]
    n = len(refs)
    if task == "gsm8k":
        ok = 0
        for p, r in zip(preds, refs):
            a, b = gsm_ans(p), gsm_ans(r)
            ok += a is not None and b is not None and abs(a - b) < 1e-6
        return ok / n
    if task == "wikisql":
        return sum(_norm_sql(p) == _norm_sql(r) for p, r in zip(preds, refs)) / n
    if task == "xsum":
        from rouge_score import rouge_scorer
        sc = rouge_scorer.RougeScorer(["rougeL"])
        return sum(sc.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)) / n
    if task == "pubmedqa":
        def first(s):
            w = re.findall(r"[a-z]+", s.lower())
            return w[0] if w else ""
        return sum(first(p) == first(r) for p, r in zip(preds, refs)) / n
    raise ValueError(task)

In [ ]:
%%writefile sync.py
"""
Exchange files between machines through a PRIVATE Hugging Face dataset repo.

Environment (set in the notebook):
  HF_TOKEN  - Hugging Face token with WRITE permission
  S2L_REPO  - 'username/repo-name' (same on every machine; created automatically, private)

Usage (paths are relative to the project folder; a path can be a file or a folder):
  python sync.py push data/gsm8k.json buffers trajectories
  python sync.py pull data buffers
  python sync.py wait data/gsm8k.json data/xsum.json [--timeout 12] [--poll 60]
      -> blocks until every listed file exists in the repo (timeout in hours)
"""
import argparse
import os
import sys
import time

from huggingface_hub import HfApi, snapshot_download


def _cfg():
    repo, tok = os.environ.get("S2L_REPO", ""), os.environ.get("HF_TOKEN", "")
    if "/" not in repo:
        sys.exit("Set S2L_REPO='username/repo-name' (same on every machine).")
    if not tok:
        sys.exit("Set HF_TOKEN (needs WRITE permission).")
    return repo, tok, HfApi(token=tok)


def _norm(p):
    return os.path.normpath(p).replace("\\", "/")


def _retry(fn, tries=4):
    for i in range(tries):
        try:
            return fn()
        except Exception as e:  # noqa: BLE001
            if i == tries - 1:
                raise
            print(f"  retry {i + 1}/{tries - 1}: {e}")
            time.sleep(5 * (i + 1))


def push(paths):
    repo, _, api = _cfg()
    _retry(lambda: api.create_repo(repo, repo_type="dataset", private=True, exist_ok=True))
    for p in map(_norm, paths):
        if os.path.isdir(p):
            _retry(lambda p=p: api.upload_folder(folder_path=p, path_in_repo=p, repo_id=repo,
                                                 repo_type="dataset", commit_message=f"push {p}"))
        elif os.path.isfile(p):
            _retry(lambda p=p: api.upload_file(path_or_fileobj=p, path_in_repo=p, repo_id=repo,
                                               repo_type="dataset", commit_message=f"push {p}"))
        else:
            print(f"  skip (not found): {p}")
            continue
        print(f"  pushed {p}")


def pull(paths):
    repo, tok, _ = _cfg()
    pats = []
    for p in map(_norm, paths):
        pats += [p, p + "/*"]  # file or whole folder
    _retry(lambda: snapshot_download(repo_id=repo, repo_type="dataset", allow_patterns=pats,
                                     local_dir=".", token=tok))
    print(f"  pulled {', '.join(map(_norm, paths))}")


def wait(files, timeout_h, poll):
    repo, _, api = _cfg()
    files = [_norm(f) for f in files]
    end, last = time.time() + timeout_h * 3600, None
    while True:
        try:
            have = set(api.list_repo_files(repo, repo_type="dataset"))
        except Exception as e:  # repo may not exist yet
            have = set()
            print(f"  repo not readable yet ({type(e).__name__})")
        missing = [f for f in files if f not in have]
        if not missing:
            print("  all files present")
            return
        if time.time() > end:
            sys.exit(f"timed out waiting for: {missing}")
        if missing != last:
            print(f"{time.strftime('%H:%M:%S')} waiting for {missing}")
            last = missing
        time.sleep(poll)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    for c in ("push", "pull"):
        sub.add_parser(c).add_argument("paths", nargs="+")
    w = sub.add_parser("wait")
    w.add_argument("files", nargs="+")
    w.add_argument("--timeout", type=float, default=12.0, help="hours")
    w.add_argument("--poll", type=int, default=60, help="seconds between checks")
    a = ap.parse_args()
    {"push": lambda: push(a.paths), "pull": lambda: pull(a.paths),
     "wait": lambda: wait(a.files, a.timeout, a.poll)}[a.cmd]()

In [ ]:
%%writefile s2l_lib.py
"""
Shared code for the two S2L-replay stage scripts:
    s2l_stage12.py   stages 1-2 (GSM8K, then WikiSQL + S2L buffer of GSM8K)  -> uploads a hand-off to Hugging Face
    s2l_stage34.py   downloads that hand-off, then stages 3-4 (XSum, then PubMedQA, each with S2L buffers)

Both scripts use THIS file for training and evaluation, so the two halves of the run are trained and
scored identically. Only S2L replay is supported (buffers/s2l_<ratio>/<task>.json from select_s2l.py).

What is handed over through the Hugging Face Hub (a folder inside a dataset repo, env HF_FOLDER, default 's2l_handoff/'):
    adapters/<run>/after_gsm8k, after_wikisql   the LoRA adapter after each finished stage
    results/<run>/results.json                  scores, held-out losses, timing and training-loss logs so far
    buffers/s2l_<ratio>/*.json                  the S2L buffers (indices into data/<task>.json)
    data/*.json                                 the exact cached train/test rows the indices refer to
    handoff/<run>.json                          manifest: hyper-parameters, stages done, SHA-256 of every file
    handoff/common.py                           a copy of the sender's common.py, to compare with yours

Authentication: set the HF_TOKEN environment variable, or add a Colab secret named HF_TOKEN, or run
`huggingface-cli login`. Pushing needs a token with WRITE access; pulling a private repo needs read access
to it (add the teammate as a collaborator, or make the repo public).
"""
import argparse
import fnmatch
import hashlib
import json
import math
import os
import random
import time
from datetime import datetime, timezone

import torch
import torch.nn.functional as F
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainerCallback, TrainingArguments

import common as C

MAXLEN = 768
EFFECTIVE_BS = 16                                         # same as the baselines: 8 x 2
MICRO_BS = int(os.environ.get("S2L_MICRO_BS", "2"))       # sequences per GPU step; lower it if you hit CUDA OOM (T4)
assert EFFECTIVE_BS % MICRO_BS == 0, "S2L_MICRO_BS must divide 16 (1, 2, 4, 8)"


# ============================================================================ data / eval helpers
class SFTData(torch.utils.data.Dataset):
    def __init__(self, rows, tok):
        self.items = [C.encode(tok, r, MAXLEN) for r in rows]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


class LossLogger(TrainerCallback):
    """Records (step, epoch, training loss, seconds since the stage started) at every logging step."""

    def __init__(self):
        self.t0, self.log = time.time(), []

    def on_train_begin(self, args, state, control, **kw):
        self.t0, self.log = time.time(), []

    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs:
            self.log.append({"step": state.global_step, "epoch": round(float(logs.get("epoch", 0)), 3),
                             "loss": round(float(logs["loss"]), 4), "time_s": round(time.time() - self.t0, 1)})


@torch.no_grad()
def generate(model, tok, prompts, max_new, bs=16):
    model.eval()
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(prompts), bs):
        batch = [C.build_prompt(p) for p in prompts[i:i + bs]]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]


def eval_all(model, tok, tests):
    res = {}
    for t, rows in tests.items():
        preds = generate(model, tok, [r["prompt"] for r in rows], C.TASKS[t]["max_new"])
        res[t] = C.score(t, preds, [r["response"] for r in rows])
        print(f"    eval {t}: {res[t]:.3f}   e.g. pred={preds[0][:80]!r}")
    return res


@torch.no_grad()
def test_loss(model, tok, rows, bs=2):
    """Token-weighted mean loss on the RESPONSE tokens of held-out rows (teacher forced)."""
    model.eval()
    tot, n = 0.0, 0
    for i in range(0, len(rows), bs):
        b = C.collate([C.encode(tok, r, MAXLEN) for r in rows[i:i + bs]], tok.pad_token_id)
        b = {k: v.to(model.device) for k, v in b.items()}
        logits = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits.float()[:, :-1]
        lab = b["labels"][:, 1:]
        tot += F.cross_entropy(logits.reshape(-1, logits.size(-1)), lab.reshape(-1),
                               ignore_index=-100, reduction="sum").item()
        n += int((lab != -100).sum())
    return tot / max(n, 1)


def evaluate(model, tok, tests):
    """Task scores + held-out loss on every task, and how long evaluating took (seconds)."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()          # give back training memory before generating
    t0 = time.time()
    scores = eval_all(model, tok, tests)
    losses = {t: test_loss(model, tok, rows) for t, rows in tests.items()}
    print("    test loss: " + "  ".join(f"{t} {v:.3f}" for t, v in losses.items()))
    return scores, losses, time.time() - t0


# ============================================================================ S2L buffers
def run_name(ratio, seed):
    return f"s2l_{ratio}_s{seed}"


def buffer_path(root, ratio, task):
    return f"{root}/buffers/s2l_{ratio}/{task}.json"


def make_s2l_buffer(task, rows, ratio, root):
    """The S2L-selected replay rows of an earlier task (indices saved by select_s2l.py)."""
    path = buffer_path(root, ratio, task)
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Run `python select_s2l.py --ratios {ratio}` first "
                                f"(or pull the buffers from the Hugging Face hand-off).")
    with open(path) as f:
        idx = json.load(f)
    assert max(idx) < len(rows), f"buffer index out of range for {task}: data/ and buffers/ are out of sync"
    return [rows[i] for i in idx]


def missing_buffers(order, ratio, root):
    """Tasks (all but the last) whose S2L buffer file does not exist yet."""
    return [t for t in order[:-1] if not os.path.exists(buffer_path(root, ratio, t))]


# ============================================================================ model + one training stage
def build_model(resume_dir=None):
    """Fresh LoRA on the base model, or continue (trainable) from a saved adapter directory."""
    tok = AutoTokenizer.from_pretrained(C.TARGET_MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    model = AutoModelForCausalLM.from_pretrained(
        C.TARGET_MODEL, torch_dtype=torch.bfloat16 if use_bf16 else torch.float16, device_map="auto")
    if resume_dir:
        model = PeftModel.from_pretrained(model, resume_dir, is_trainable=True)
        print(f"resumed LoRA adapter from {resume_dir}")
    else:
        model = get_peft_model(model, LoraConfig(
            r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    if not use_bf16:  # fp16 GPUs (e.g. Colab T4): keep trainable LoRA weights in fp32
        for p in model.parameters():
            if p.requires_grad:
                p.data = p.data.float()
    model.print_trainable_parameters()
    return model, tok, use_bf16


def train_stage(model, tok, rows, out_dir, epochs, lr, seed, debug, use_bf16):
    """Train the current adapter on `rows` for one stage. Returns timing and the training-loss log."""
    n_epochs = 1 if debug else epochs
    total_steps = math.ceil(n_epochs * math.ceil(len(rows) / EFFECTIVE_BS))
    warmup = max(1, math.ceil(0.03 * total_steps))      # = warmup_ratio 0.03 (transformers 5 removed that argument)
    targs = TrainingArguments(
        output_dir=out_dir, per_device_train_batch_size=MICRO_BS, gradient_accumulation_steps=EFFECTIVE_BS // MICRO_BS,
        learning_rate=lr, num_train_epochs=n_epochs,
        lr_scheduler_type="cosine", warmup_steps=warmup, logging_steps=1 if debug else 20,
        save_strategy="no", report_to="none", bf16=use_bf16, fp16=not use_bf16,
        remove_unused_columns=False, seed=seed)
    cb = LossLogger()
    trainer = Trainer(model=model, args=targs, train_dataset=SFTData(rows, tok), callbacks=[cb],
                      data_collator=lambda b: C.collate(b, tok.pad_token_id))
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    trainer.train()
    train_s = time.time() - t0
    print(f"  trained in {train_s / 60:.1f} min ({trainer.state.global_step} steps)")
    peak = round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None
    return {"train_s": train_s, "steps": trainer.state.global_step, "log": cb.log, "peak_mem_gb": peak}


# ============================================================================ the stage loop
def run_stages(order, k_first, k_last, ratio, seed, epochs, lr, debug, resume):
    """
    Train stages k_first..k_last (0-based indices into `order`) with S2L replay.
    resume=False: start from a fresh LoRA and evaluate the zero-shot base model first (stage 1 onwards).
    resume=True:  continue from adapters/<run>/after_<order[k_first-1]> and results/<run>/results.json.
    Writes results/<run>/results.json after every stage (same format as sequential_finetune.py).
    """
    R_ = C.root(debug)
    name = run_name(ratio, seed)
    res_path = f"{R_}/results/{name}/results.json"
    os.makedirs(os.path.dirname(res_path), exist_ok=True)
    random.seed(seed)
    torch.manual_seed(seed)

    resume_dir = None
    if resume:
        resume_dir = f"{R_}/adapters/{name}/after_{order[k_first - 1]}"
        assert os.path.isdir(resume_dir), f"missing adapter {resume_dir}"
    model, tok, use_bf16 = build_model(resume_dir)

    train, test = {}, {}
    for t in order:
        train[t], test[t] = C.load_task(t, debug=debug)
        print(f"{t}: {len(train[t])} train / {len(test[t])} test")
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    print(f"GPU: {gpu}")

    if resume:
        with open(res_path) as f:
            prev = json.load(f)
        assert prev["order"] == order, f"task order differs from the hand-off: {prev['order']} vs {order}"
        assert prev.get("replay") == "s2l", "the hand-off was not an S2L run"
        for key, val in (("seed", seed), ("epochs", epochs), ("lr", lr), ("ratio", ratio)):
            assert prev.get(key, val) == val, f"{key} differs from the hand-off ({prev.get(key)} vs {val})"
        assert order[k_first - 1] in prev["R"], f"{res_path} has no results for {order[k_first - 1]}"
        R, L, S = prev["R"], prev.get("L", {}), prev.get("stages", {})
        print(f"resumed results: done = {[k for k in R if k != 'base']}")
    else:
        # LoRA B matrices start at zero, so this equals the zero-shot base model
        print("\n=== zero-shot base model ===")
        R, L, S = {}, {}, {}
        R["base"], L["base"], _ = evaluate(model, tok, test)

    for k in range(k_first, k_last + 1):
        t = order[k]
        rows = list(train[t])
        for earlier in order[:k]:
            rows += make_s2l_buffer(earlier, train[earlier], ratio, R_)
        random.Random(seed).shuffle(rows)
        print(f"\n=== Stage {k + 1}/{len(order)}: {t} | {len(train[t])} new + {len(rows) - len(train[t])} replay rows ===")

        info = train_stage(model, tok, rows, f"{R_}/tmp/{name}", epochs, lr, seed, debug, use_bf16)
        model.save_pretrained(f"{R_}/adapters/{name}/after_{t}")
        R[t], L[t], eval_s = evaluate(model, tok, test)
        last = [e["loss"] for e in info["log"][-3:]]
        S[t] = {"n_new": len(train[t]), "n_replay": len(rows) - len(train[t]), "n_rows": len(rows),
                "steps": info["steps"], "train_time_s": round(info["train_s"], 1), "eval_time_s": round(eval_s, 1),
                "final_train_loss": round(sum(last) / len(last), 4) if last else None,
                "peak_mem_gb": info["peak_mem_gb"], "gpu": gpu, "log": info["log"]}
        with open(res_path, "w") as f:
            json.dump({"order": order, "replay": "s2l", "ratio": ratio, "seed": seed, "epochs": epochs, "lr": lr,
                       "gpu": "/".join(sorted({x.get("gpu", gpu) for x in S.values()} | {gpu})),
                       "R": R, "L": L, "stages": S}, f, indent=2)

    print_summary(R, S, order, gpu)
    return R, L, S, name


def print_summary(R, S, order, gpu):
    print("\nResults matrix (rows = after training on, cols = evaluated on)")
    print("after".ljust(10) + "".join(c.ljust(10) for c in order))
    for r, v in R.items():
        print(r.ljust(10) + "".join(f"{v[c]:.3f}".ljust(10) if c in v else "-".ljust(10) for c in order))
    print("\nTraining time per stage")
    for t, st in S.items():
        fl = f"{st['final_train_loss']:.3f}" if st["final_train_loss"] is not None else "n/a"
        print(f"  {t.ljust(9)} {st['n_rows']:>6} rows {st['steps']:>5} steps  train {st['train_time_s'] / 60:6.1f} min  "
              f"eval {st['eval_time_s'] / 60:5.1f} min  final train loss {fl}")
    print(f"  total train {sum(x['train_time_s'] for x in S.values()) / 60:.1f} min  on {gpu}")


# ============================================================================ hand-off: files + manifest
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _p(*parts):
    return os.path.normpath(os.path.join(*parts))


def manifest_path(root, run):
    return _p(root, "handoff", f"{run}.json")


def _data_paths(order):
    return [_p(C.DATA_DIR, f"{t}.json") for t in order]


def _versions():
    from importlib.metadata import version, PackageNotFoundError
    out = {}
    for pkg in ("torch", "transformers", "peft", "datasets"):
        try:
            out[pkg] = version(pkg)
        except PackageNotFoundError:
            out[pkg] = None
    return out


def write_manifest(run, ratio, seed, epochs, lr, order, done, root, include_data, gpu):
    """Describe the hand-off, including the SHA-256 of the buffers, the data and common.py."""
    hashed = [buffer_path(root, ratio, t) for t in order[:-1]] + (_data_paths(order) if include_data else [])
    manifest = {
        "run": run, "ratio": ratio, "seed": seed, "epochs": epochs, "lr": lr, "order": order,
        "stages_done": done, "target_model": C.TARGET_MODEL, "gpu": gpu, "versions": _versions(),
        "pushed_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "common_sha256": sha256("common.py"),
        "files_sha256": {_p(p): sha256(p) for p in hashed},
    }
    path = manifest_path(root, run)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(manifest, f, indent=2)
    return manifest


def handoff_files(run, ratio, order, root, include_data):
    """(local_path, repo_path) pairs that someone else needs to continue this run."""
    local = []
    for d, _, names in os.walk(_p(root, "adapters", run)):
        local += [os.path.join(d, n) for n in names]
    local.append(_p(root, "results", run, "results.json"))
    local += [buffer_path(root, ratio, t) for t in order[:-1]]
    if include_data:
        local += _data_paths(order)
    local.append(manifest_path(root, run))
    files = [(_p(p), _p(p)) for p in local]            # repo path = local relative path
    files.append(("common.py", _p(root, "handoff", "common.py")))
    return files


def final_files(run, root):
    """What stage 3-4 uploads when it is done: all adapters and the merged results."""
    local = []
    for d, _, names in os.walk(_p(root, "adapters", run)):
        local += [os.path.join(d, n) for n in names]
    local.append(_p(root, "results", run, "results.json"))
    return [(_p(p), _p(p)) for p in local]


def pull_patterns(run, ratio, root, include_data):
    pats = [_p(root, "adapters", run) + "/*", _p(root, "results", run) + "/*", manifest_path(root, run),
            _p(root, "handoff", "common.py"), _p(root, "buffers", f"s2l_{ratio}") + "/*"]
    if include_data:
        pats.append(_p(C.DATA_DIR) + "/*")
    return pats


def verify_handoff(run, ratio, root, include_data, order_expected=None):
    """Check a pulled hand-off against the local common.py and the files that came with it."""
    path = manifest_path(root, run)
    assert os.path.exists(path), f"{path} not found - pull the hand-off first"
    with open(path) as f:
        m = json.load(f)
    problems = []
    if m["ratio"] != ratio:
        problems.append(f"ratio in the hand-off is {m['ratio']}, you asked for {ratio}")
    if order_expected is not None and m["order"] != order_expected:
        problems.append(f"task order differs: hand-off {m['order']} vs your common.py {order_expected}")
    if m["common_sha256"] != sha256("common.py"):
        problems.append("your common.py differs from the sender's (different data sizes, prompts or metrics would "
                        f"make the run inconsistent). The sender's copy is in {_p(root, 'handoff', 'common.py')}: "
                        "compare it with yours and use the sender's.")
    for p, h in m["files_sha256"].items():
        if not os.path.exists(p):
            problems.append(f"missing file {p}")
        elif sha256(p) != h:
            problems.append(f"checksum mismatch for {p} (corrupted or different version)")
    if problems:
        raise SystemExit("Hand-off check FAILED:\n  - " + "\n  - ".join(problems))
    mine = _versions()
    for pkg, v in m.get("versions", {}).items():
        if v and mine.get(pkg) and v.split(".")[:2] != mine[pkg].split(".")[:2]:
            print(f"note: {pkg} was {v} for stages 1-2 and is {mine[pkg]} here (usually harmless)")
    return m


# ============================================================================ Hugging Face Hub
HF_REPO_TYPE = "dataset"      # the hand-off lives in the SAME dataset repo as the exchange files


def hf_folder():
    """Folder inside the repo that holds the hand-off (env HF_FOLDER, default 's2l_handoff')."""
    return os.environ.get("HF_FOLDER", "s2l_handoff").strip("/")


def hf_token():
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:  # noqa: BLE001 (not on Colab, or no such secret)
            tok = None
    if not tok:
        try:
            from huggingface_hub import get_token
            tok = get_token()
        except Exception:  # noqa: BLE001
            tok = None
    return tok


def _hf_preflight(repo_id, private=True):
    """Fail early (before hours of training) if the token or repo is not usable. Creates the repo if needed."""
    from huggingface_hub import HfApi
    token = hf_token()
    if not token:
        raise SystemExit("No Hugging Face token found. Set HF_TOKEN (env var or Colab secret) with WRITE access, "
                         "or run `huggingface-cli login`.")
    from huggingface_hub import CommitOperationAdd
    api = HfApi(token=token)
    who = api.whoami()["name"]
    api.create_repo(repo_id, repo_type=HF_REPO_TYPE, private=private, exist_ok=True)
    # Prove WRITE access now: a read-only token or a repo you cannot write to would otherwise only fail
    # after hours of training. (Commits a 2-byte marker file inside the hand-off folder; harmless.)
    api.create_commit(repo_id=repo_id, repo_type=HF_REPO_TYPE, commit_message="preflight: write access check",
                      operations=[CommitOperationAdd(path_in_repo=f"{hf_folder()}/handoff/.preflight",
                                                     path_or_fileobj=b"ok")])
    print(f"Hugging Face OK: logged in as {who}; write access to {repo_id} (folder {hf_folder()}/)")


def _hf_push(repo_id, files, message, private=True):
    from huggingface_hub import CommitOperationAdd, HfApi
    api = HfApi(token=hf_token())
    api.create_repo(repo_id, repo_type=HF_REPO_TYPE, private=private, exist_ok=True)
    pre = hf_folder()
    ops = [CommitOperationAdd(path_in_repo=f"{pre}/{rp.replace(os.sep, '/')}", path_or_fileobj=lp) for lp, rp in files]
    api.create_commit(repo_id=repo_id, repo_type=HF_REPO_TYPE, operations=ops, commit_message=message)


def _hf_pull(repo_id, patterns):
    """Download <folder>/<pattern> files and place them at <pattern> in the current directory."""
    import shutil
    import tempfile
    from huggingface_hub import snapshot_download
    pre = hf_folder()
    with tempfile.TemporaryDirectory() as tmp:
        snapshot_download(repo_id=repo_id, repo_type=HF_REPO_TYPE, local_dir=tmp, token=hf_token(),
                          allow_patterns=[f"{pre}/{p.replace(os.sep, '/')}" for p in patterns])
        src = os.path.join(tmp, pre)
        if not os.path.isdir(src):
            raise SystemExit(f"Nothing found under {pre}/ in {repo_id}. Has Part A pushed the hand-off "
                             f"(same HF_FOLDER, same repo)?")
        shutil.copytree(src, ".", dirs_exist_ok=True)


def preflight(repo_id, private=True):
    _hf_preflight(repo_id, private)


def push_handoff(repo_id, run, ratio, seed, epochs, lr, order, done, debug, gpu, private=True):
    root = C.root(debug)
    include_data = not debug      # debug runs slice the cached data; they never ship it
    write_manifest(run, ratio, seed, epochs, lr, order, done, root, include_data, gpu)
    files = handoff_files(run, ratio, order, root, include_data)
    size_mb = sum(os.path.getsize(lp) for lp, _ in files) / 1e6
    print(f"uploading {len(files)} files ({size_mb:.1f} MB) to {repo_id} ...")
    _hf_push(repo_id, files, f"hand-off of {run} after {done[-1]}", private)
    print(f"done. Teammate: python s2l_stage34.py --hf_repo {repo_id} --ratio {ratio} --seed {seed}"
          + (" --debug" if debug else ""))


def push_existing(repo_id, ratio, seed, order, debug, private=True):
    """Upload a finished stages 1-2 run that is already on disk (e.g. the upload failed after training)."""
    root = C.root(debug)
    run = run_name(ratio, seed)
    path = f"{root}/results/{run}/results.json"
    assert os.path.exists(path), f"{path} not found: nothing to upload"
    with open(path) as f:
        r = json.load(f)
    assert r["order"] == order, f"task order in {path} differs from common.py"
    done = [t for t in order if t in r["R"]]
    assert done == order[:2], (f"{path} has stages {done}; expected exactly {order[:2]} for a stages 1-2 hand-off "
                               f"(if later stages are already done there is nothing to hand over)")
    push_handoff(repo_id, run, ratio, seed, r["epochs"], r["lr"], order, done, debug, r.get("gpu", "?"), private)


def pull_handoff(repo_id, run, ratio, debug):
    root = C.root(debug)
    print(f"downloading hand-off of {run} from {repo_id} ...")
    _hf_pull(repo_id, pull_patterns(run, ratio, root, include_data=not debug))


def push_final(repo_id, run, debug, private=True):
    root = C.root(debug)
    files = final_files(run, root)
    print(f"uploading final adapters + results ({len(files)} files) to {repo_id} ...")
    _hf_push(repo_id, files, f"final results of {run}", private)


def pull_results(repo_id, run, debug=False):
    """Fetch the adapters and results.json of a run (e.g. to run aggregate_results.py on another machine)."""
    root = C.root(debug)
    _hf_pull(repo_id, [_p(root, "adapters", run) + "/*", _p(root, "results", run) + "/*"])
    print(f"pulled results/{run}/results.json")


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description="Fetch the results of a run from the Hugging Face repo")
    ap.add_argument("--hf_repo", required=True)
    ap.add_argument("--ratio", type=float, default=0.02)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--debug", action="store_true")
    a = ap.parse_args()
    pull_results(a.hf_repo, run_name(a.ratio, a.seed), a.debug)

In [ ]:
%%writefile s2l_stage12.py
"""
Stages 1-2 of the continual run, S2L replay only  (person A).

    stage 1: GSM8K                              (nothing to replay yet)
    stage 2: WikiSQL + S2L buffer of GSM8K

It first evaluates the zero-shot base model, evaluates on ALL tasks after each stage, and then uploads
everything person B needs to run stages 3-4 to a folder of a Hugging Face repo (see s2l_lib.py for the list).

Before you start you need (from select_s2l.py): data/*.json and buffers/s2l_<ratio>/<task>.json for
GSM8K, WikiSQL and XSum. The script checks this and the Hugging Face login BEFORE training.

Usage:
    python s2l_stage12.py --debug --hf_repo YOUR_NAME/s2l-continual        # smoke test (writes to ./debug)
    python s2l_stage12.py --hf_repo YOUR_NAME/s2l-continual --ratio 0.02
    python s2l_stage12.py --ratio 0.02 --no_push                            # train only, upload later
    python s2l_stage12.py --hf_repo YOUR_NAME/s2l-continual --push_only     # no training: upload what is on disk
"""
import argparse

import common as C
import s2l_lib as L


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--hf_repo", default=None, help="Hugging Face repo to hand over to, e.g. user/s2l-continual")
    ap.add_argument("--ratio", type=float, default=0.02, help="S2L buffer size (fraction of each task)")
    ap.add_argument("--epochs", type=float, default=2)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--debug", action="store_true", help="50 train / 20 test examples, 1 epoch, writes to ./debug")
    ap.add_argument("--no_push", action="store_true", help="skip the Hugging Face upload")
    ap.add_argument("--public", action="store_true", help="create the repo as public (default: private)")
    ap.add_argument("--push_only", action="store_true",
                    help="do not train: upload the stages 1-2 run already on disk (use if the upload failed)")
    args = ap.parse_args()

    order = C.ORDER
    assert len(order) >= 3, "need at least 3 tasks to split into stages 1-2 and the rest"
    if (args.push_only or not args.no_push) and not args.hf_repo:
        ap.error("--hf_repo is required (or pass --no_push)")
    R_ = C.root(args.debug)

    # ---- fail early, before hours of training
    missing = L.missing_buffers(order, args.ratio, R_)
    if missing:
        raise SystemExit(f"Missing S2L buffers for {missing} at ratio {args.ratio} in {R_}/buffers/s2l_{args.ratio}/. "
                         f"Stage 3-4 needs them too. Run: python select_s2l.py --ratios {args.ratio}"
                         + (" --debug" if args.debug else ""))
    if not args.no_push:
        L.preflight(args.hf_repo, private=not args.public)
    if args.push_only:
        L.push_existing(args.hf_repo, args.ratio, args.seed, order, args.debug, private=not args.public)
        return

    R, _, S, name = L.run_stages(order, k_first=0, k_last=1, ratio=args.ratio, seed=args.seed,
                                 epochs=args.epochs, lr=args.lr, debug=args.debug, resume=False)

    if args.no_push:
        print(f"\nStages 1-2 done (run {name}); not uploaded (--no_push).")
        return
    gpu = next(iter(S.values()))["gpu"]
    L.push_handoff(args.hf_repo, name, args.ratio, args.seed, args.epochs, args.lr, order,
                   done=order[:2], debug=args.debug, gpu=gpu, private=not args.public)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile s2l_stage34.py
"""
Stages 3-4 of the continual run, S2L replay only  (person B).

    stage 3: XSum     + S2L buffers of GSM8K and WikiSQL
    stage 4: PubMedQA + S2L buffers of GSM8K, WikiSQL and XSum

It downloads person A's hand-off from the Hugging Face repo (LoRA adapter after stage 2, results so far,
S2L buffers, the cached data and a manifest), checks that it matches your common.py and that no file is
corrupted, then continues training from that adapter. epochs / lr / seed / ratio come from person A's run
so both halves are trained with identical settings. At the end it evaluates on ALL tasks, writes the merged
results/<run>/results.json (the same format as a single uninterrupted run, so aggregate_results.py works
on it) and uploads the final adapters + results to the same repo.

Usage (use the same --ratio and --seed as person A):
    python s2l_stage34.py --hf_repo YOUR_NAME/s2l-continual --ratio 0.02 --seed 0
    python s2l_stage34.py --debug --hf_repo YOUR_NAME/s2l-continual         # smoke test of the hand-off
    python s2l_stage34.py --hf_repo YOUR_NAME/s2l-continual --skip_pull      # files are already local
    python s2l_stage34.py --hf_repo YOUR_NAME/s2l-continual --push_only      # no training: upload the final results on disk

Requirement: your common.py must be identical to person A's (the script checks this).
"""
import argparse
import json

import common as C
import s2l_lib as L


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--hf_repo", required=True, help="the repo person A pushed to, e.g. user/s2l-continual")
    ap.add_argument("--ratio", type=float, default=0.02, help="must match person A's --ratio")
    ap.add_argument("--seed", type=int, default=0, help="must match person A's --seed")
    ap.add_argument("--debug", action="store_true", help="smoke test: use the ./debug hand-off")
    ap.add_argument("--skip_pull", action="store_true", help="do not download; use the files already on disk")
    ap.add_argument("--no_push", action="store_true", help="do not upload the final results")
    ap.add_argument("--public", action="store_true", help="only matters if the repo has to be created")
    ap.add_argument("--push_only", action="store_true",
                    help="do not train: upload the finished run already on disk (use if the final upload failed)")
    args = ap.parse_args()

    order = C.ORDER
    name = L.run_name(args.ratio, args.seed)
    R_ = C.root(args.debug)

    if args.push_only:
        L.preflight(args.hf_repo, private=not args.public)
        with open(f"{R_}/results/{name}/results.json") as f:
            finished = json.load(f)["R"]
        done = [t for t in order if t in finished]
        assert done == order, f"run {name} on disk has stages {done}; expected all of {order}"
        L.push_final(args.hf_repo, name, args.debug, private=not args.public)
        return

    if not args.skip_pull:
        L.pull_handoff(args.hf_repo, name, args.ratio, args.debug)
    m = L.verify_handoff(name, args.ratio, R_, include_data=not args.debug, order_expected=order)
    assert m["stages_done"] == order[:2], f"expected a hand-off after stages 1-2, got {m['stages_done']}"
    print(f"hand-off OK: run {name}, stages done {m['stages_done']}, pushed {m['pushed_at']} "
          f"on {m['gpu']}; epochs={m['epochs']} lr={m['lr']}")
    if not args.no_push:
        L.preflight(args.hf_repo, private=not args.public)

    R, _, S, name = L.run_stages(order, k_first=2, k_last=len(order) - 1, ratio=m["ratio"], seed=m["seed"],
                                 epochs=m["epochs"], lr=m["lr"], debug=args.debug, resume=True)

    if not args.no_push:
        L.push_final(args.hf_repo, name, args.debug, private=not args.public)
    final = {t: round(R[order[-1]][t], 3) for t in order}
    print(f"\nFinished run {name}. Final scores after stage {len(order)}: {json.dumps(final)}")
    print(f"Merged results: {R_}/results/{name}/results.json  ->  python aggregate_results.py"
          + (" --root debug" if args.debug else ""))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile aggregate_results.py
"""
Stage 3 - Aggregate results/<run>/results.json into tables and a plot.

Runs named like  s2l_0.05_s0, s2l_0.05_s1  are grouped (mean +- std over seeds) as  s2l_0.05.

Metrics (order = [T1..Tn], R[a][b] = score on task b after training through task a):
  avg final  = mean_j R[Tn][Tj]
  forgetting = mean over j<n of ( max_{j<=l<=n-1} R[Tl][Tj]  -  R[Tn][Tj] )   (lower is better)
  BWT        = mean over j<n of ( R[Tn][Tj] - R[Tj][Tj] )                     (higher is better)

If the runs were made with the timing/loss logging (results.json has "stages" and "L"), it also writes:
  results/summary_timing.csv       time per stage, total time, final test loss
  results/train_loss_by_stage.png  training loss vs step, one panel per stage
  results/train_loss_vs_time.png   training loss vs cumulative wall-clock minutes
  results/time_per_stage.png       training minutes per stage (rows annotated)
  results/test_loss_forgetting.png held-out loss on each task after every stage
  results/time_vs_quality.png      total training time vs final score / final test loss
  results/summary_stages.csv       per method and stage: rows, steps, times, losses, time to converge, peak GPU memory
  results/convergence_time.png     time (min) until the training loss reached --conv_frac of its total drop, per stage
  results/peak_memory.png          peak GPU memory (GB) during training, per stage
  results/method_comparison.png    one figure: final score, final held-out loss, training / convergence time, peak memory

"Time to converge" (per stage): training loss is smoothed with a 3-point trailing mean; the stage has converged at the
first logged point where the smoothed loss has covered --conv_frac (default 0.90) of the way from its first value to its
final value. Reported as minutes since the stage started (and optimizer steps). Peak memory is torch's
max_memory_allocated during the training of that stage (evaluation is not included).

Usage:  python aggregate_results.py [--root .]
"""
import argparse
import csv
import glob
import json
import os
import re
from collections import defaultdict

import numpy as np


def metrics(R, order):
    T = len(order)
    last = R[order[-1]]
    avg = float(np.mean([last[t] for t in order]))
    fgt = float(np.mean([max(R[order[l]][order[j]] for l in range(j, T - 1)) - last[order[j]]
                         for j in range(T - 1)]))
    bwt = float(np.mean([last[order[j]] - R[order[j]][order[j]] for j in range(T - 1)]))
    return avg, fgt, bwt


def ms(xs):
    return f"{np.mean(xs):.3f} ± {np.std(xs):.3f}" if len(xs) > 1 else f"{xs[0]:.3f}"


def msn(xs):
    """mean ± std ignoring None/NaN; 'n/a' if nothing is left."""
    xs = [float(x) for x in xs if x is not None and not np.isnan(x)]
    return ms(xs) if xs else "n/a"


def convergence(st, frac=0.9):
    """(time_s, step) at which the smoothed training loss covered `frac` of its total drop; None if too few logs."""
    log = st.get("log") or []
    if len(log) < 2:
        return None
    loss = np.array([e["loss"] for e in log], float)
    w = min(3, len(loss))
    sm = np.convolve(loss, np.ones(w) / w, mode="valid")      # sm[i] = mean(loss[i:i+w]) -> log index i+w-1
    drop = sm[0] - sm[-1]
    idx = 0 if drop <= 0 else int(np.argmax(sm <= sm[-1] + (1 - frac) * drop))
    e = log[idx + w - 1]
    return {"time_s": float(e["time_s"]), "step": int(e["step"])}


def _conv_min(r, t, frac):
    st = r["stages"].get(t)
    c = convergence(st, frac) if st else None
    return c["time_s"] / 60 if c else np.nan


def _peak(r):
    v = [r["stages"][t].get("peak_mem_gb") for t in r["stages"]]
    v = [x for x in v if x is not None]
    return max(v) if v else np.nan


def _runs_with_stages(meta):
    return {m: [r for r in runs if r.get("stages")] for m, runs in meta.items()
            if any(r.get("stages") for r in runs)}


def timing_table(meta, order, frac=0.9):
    header = (["method", "seeds"] + [f"train_min_{t}" for t in order] +
              ["total_train_min", "total_conv_min", "total_eval_min", "avg_final_test_loss", "peak_mem_gb", "gpu"])
    rows = []
    for m, runs in _runs_with_stages(meta).items():
        per = [[r["stages"][t]["train_time_s"] / 60 if t in r["stages"] else np.nan for r in runs] for t in order]
        tot = [sum(r["stages"][t]["train_time_s"] for t in order if t in r["stages"]) / 60 for r in runs]
        ev = [sum(r["stages"][t]["eval_time_s"] for t in order if t in r["stages"]) / 60 for r in runs]
        fl = [np.mean([r["L"][order[-1]][t] for t in order]) for r in runs if r.get("L", {}).get(order[-1])]
        conv = [sum(_conv_min(r, t, frac) for t in order) for r in runs]       # NaN if a stage has too few logs
        rows.append([m, str(len(runs))] + [ms(p) if not np.isnan(p).any() else "n/a" for p in per] +
                    [ms(tot), msn(conv), ms(ev), ms(fl) if fl else "n/a", msn([_peak(r) for r in runs]),
                     "/".join(sorted({r.get("gpu", "?") for r in runs}))])
    return header, rows


def stage_table(meta, order, frac=0.9):
    header = ["method", "stage", "rows", "steps", "train_min", "eval_min", "final_train_loss",
              "avg_test_loss_seen_tasks", "conv_min", "conv_steps", "peak_mem_gb"]
    rows = []
    for m, runs in _runs_with_stages(meta).items():
        for k, t in enumerate(order):
            rs = [r for r in runs if t in r["stages"]]
            if not rs:
                continue
            st = [r["stages"][t] for r in rs]
            cv = [convergence(x, frac) for x in st]
            seen = order[:k + 1]
            tl = [np.mean([r["L"][t][u] for u in seen]) for r in rs if r.get("L", {}).get(t)]
            rows.append([m, t, f"{np.mean([x['n_rows'] for x in st]):.0f}", f"{np.mean([x['steps'] for x in st]):.0f}",
                         msn([x["train_time_s"] / 60 for x in st]), msn([x["eval_time_s"] / 60 for x in st]),
                         msn([x.get("final_train_loss") for x in st]), msn(tl),
                         msn([c["time_s"] / 60 if c else None for c in cv]), msn([c["step"] if c else None for c in cv]),
                         msn([x.get("peak_mem_gb") for x in st])])
    return header, rows


def training_plots(meta, order, root, frac=0.9):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    runs_of = _runs_with_stages(meta)
    if not runs_of:
        print("\n(no timing/loss info in these runs: they were made before logging was added)")
        return
    methods = list(runs_of)
    col = {m: f"C{i}" for i, m in enumerate(methods)}
    out = f"{root}/results"
    T = len(order)

    # 1. training loss vs step, one panel per stage
    fig, axs = plt.subplots(1, T, figsize=(4 * T, 3.4), squeeze=False)
    for k, t in enumerate(order):
        ax = axs[0][k]
        for m in methods:
            for i, r in enumerate(runs_of[m]):
                st = r["stages"].get(t)
                if st and st["log"]:
                    ax.plot([e["step"] for e in st["log"]], [e["loss"] for e in st["log"]], color=col[m],
                            alpha=0.8, lw=1, label=m if i == 0 else None)
        ax.set_title(f"stage {k + 1}: {t}", fontsize=9)
        ax.set_xlabel("optimizer step")
        ax.set_ylabel("training loss") if k == 0 else None
    axs[0][0].legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(f"{out}/train_loss_by_stage.png", dpi=150)
    plt.close()

    # 2. training loss vs cumulative wall-clock time
    plt.figure(figsize=(9, 3.8))
    for m in methods:
        for i, r in enumerate(runs_of[m]):
            off = 0.0
            for j, t in enumerate(order):
                st = r["stages"].get(t)
                if not st:
                    break
                plt.plot([(off + e["time_s"]) / 60 for e in st["log"]], [e["loss"] for e in st["log"]],
                         color=col[m], alpha=0.8, lw=1, label=m if (i == 0 and j == 0) else None)
                off += st["train_time_s"]
    ref, off = runs_of[methods[0]][0], 0.0
    for k, t in enumerate(order):
        if t in ref["stages"]:
            plt.axvline(off / 60, color="gray", ls=":", lw=0.8)
            plt.text(off / 60, plt.ylim()[1], f" {t}", fontsize=7, va="top")
            off += ref["stages"][t]["train_time_s"]
    plt.xlabel("cumulative training time (min)")
    plt.ylabel("training loss")
    plt.title("Training loss vs wall-clock time (dotted lines: stage starts of the first method)", fontsize=9)
    plt.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(f"{out}/train_loss_vs_time.png", dpi=150)
    plt.close()

    # 3. training minutes per stage
    plt.figure(figsize=(8, 3.8))
    w = 0.8 / len(methods)
    x = np.arange(T)
    for i, m in enumerate(methods):
        mins = [np.nanmean([r["stages"][t]["train_time_s"] / 60 if t in r["stages"] else np.nan
                            for r in runs_of[m]]) for t in order]
        rows_ = [np.nanmean([r["stages"][t]["n_rows"] if t in r["stages"] else np.nan for r in runs_of[m]]) for t in order]
        bars = plt.bar(x - 0.4 + w * (i + 0.5), mins, w, color=col[m], label=m)
        for b, n in zip(bars, rows_):
            if not np.isnan(n):
                plt.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{int(n)}", ha="center", va="bottom", fontsize=6)
    plt.xticks(x, order)
    plt.ylabel("training time (min)")
    plt.title("Training time per stage (numbers = training rows in that stage)", fontsize=9)
    plt.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(f"{out}/time_per_stage.png", dpi=150)
    plt.close()

    # 4. held-out loss on every task after every stage (forgetting curves)
    fig, axs = plt.subplots(1, T, figsize=(4 * T, 3.4), squeeze=False)
    xl = ["base"] + order
    for j, t in enumerate(order):
        ax = axs[0][j]
        for m in methods:
            curves = []
            for r in runs_of[m]:
                L = r.get("L", {})
                if all(s_ in L for s_ in xl):
                    curves.append([L[s_][t] for s_ in xl])
            if curves:
                ax.plot(range(len(xl)), np.mean(curves, 0), marker="o", ms=3, color=col[m], label=m)
        ax.set_xticks(range(len(xl)))
        ax.set_xticklabels(xl, rotation=30, fontsize=7)
        ax.set_title(f"loss on {t}", fontsize=9)
        ax.set_xlabel("after training on")
    axs[0][0].set_ylabel("held-out loss (lower is better)")
    axs[0][0].legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(f"{out}/test_loss_forgetting.png", dpi=150)
    plt.close()

    # 5. cost vs quality
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
    for m in methods:
        for i, r in enumerate(runs_of[m]):
            if order[-1] not in r["stages"]:
                continue
            tt = sum(r["stages"][t]["train_time_s"] for t in order) / 60
            axs[0].scatter(tt, metrics(r["R"], order)[0], color=col[m], label=m if i == 0 else None)
            if r.get("L", {}).get(order[-1]):
                axs[1].scatter(tt, np.mean([r["L"][order[-1]][t] for t in order]), color=col[m])
    axs[0].set_ylabel("avg final score (higher is better)")
    axs[1].set_ylabel("avg final held-out loss (lower is better)")
    for a in axs:
        a.set_xlabel("total training time (min)")
    axs[0].legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(f"{out}/time_vs_quality.png", dpi=150)
    plt.close()

    # 6. time to converge per stage
    plt.figure(figsize=(8, 3.8))
    w = 0.8 / len(methods)
    x = np.arange(T)
    for i, m in enumerate(methods):
        v = [np.nanmean([_conv_min(r, t, frac) for r in runs_of[m]]) if any(t in r["stages"] for r in runs_of[m]) else np.nan
             for t in order]
        bars = plt.bar(x - 0.4 + w * (i + 0.5), v, w, color=col[m], label=m)
        for b, val in zip(bars, v):
            if not np.isnan(val):
                plt.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{val:.1f}", ha="center", va="bottom", fontsize=6)
    plt.xticks(x, order)
    plt.ylabel("minutes until converged")
    plt.title(f"Time to converge per stage (smoothed train loss covered {frac:.0%} of its drop)", fontsize=9)
    plt.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(f"{out}/convergence_time.png", dpi=150)
    plt.close()

    # 7. peak GPU memory per stage
    plt.figure(figsize=(8, 3.8))
    for i, m in enumerate(methods):
        v = [np.nanmean([r["stages"][t].get("peak_mem_gb") if r["stages"].get(t) and r["stages"][t].get("peak_mem_gb") is not None
                         else np.nan for r in runs_of[m]]) for t in order]
        bars = plt.bar(x - 0.4 + w * (i + 0.5), v, w, color=col[m], label=m)
        for b, val in zip(bars, v):
            if not np.isnan(val):
                plt.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{val:.2f}", ha="center", va="bottom", fontsize=6)
    plt.xticks(x, order)
    plt.ylabel("peak GPU memory during training (GB)")
    plt.title("Peak memory per stage", fontsize=9)
    plt.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(f"{out}/peak_memory.png", dpi=150)
    plt.close()

    # 8. one-figure comparison of the methods that have timing/loss logs
    fin = {m: [r for r in runs_of[m] if order[-1] in r["stages"]] for m in methods}
    fig, axs = plt.subplots(1, 4, figsize=(15, 3.4))
    xm = np.arange(len(methods))

    def bar(ax, vals_by_method, title, fmt="{:.3f}"):
        means = [np.nanmean(v) if len(v) else np.nan for v in vals_by_method]
        errs = [np.nanstd(v) if len(v) > 1 else 0 for v in vals_by_method]
        bars = ax.bar(xm, means, yerr=errs, color=[col[m] for m in methods])
        for b, val in zip(bars, means):
            if not np.isnan(val):
                ax.text(b.get_x() + b.get_width() / 2, b.get_height(), fmt.format(val), ha="center", va="bottom", fontsize=7)
        ax.set_xticks(xm)
        ax.set_xticklabels(methods, rotation=20, fontsize=7)
        ax.set_title(title, fontsize=9)

    bar(axs[0], [[metrics(r["R"], order)[0] for r in fin[m]] for m in methods], "avg final score (higher better)")
    bar(axs[1], [[np.mean([r["L"][order[-1]][t] for t in order]) for r in fin[m] if r.get("L", {}).get(order[-1])]
                 for m in methods], "avg final held-out loss (lower better)")
    tt = [[sum(r["stages"][t]["train_time_s"] for t in order) / 60 for r in fin[m]] for m in methods]
    cc = [[sum(_conv_min(r, t, frac) for t in order) for r in fin[m]] for m in methods]
    w2 = 0.38
    axs[2].bar(xm - w2 / 2, [np.nanmean(v) if len(v) else np.nan for v in tt], w2, label="total training", color="lightgray")
    axs[2].bar(xm + w2 / 2, [np.nanmean(v) if len(v) else np.nan for v in cc], w2, label="until converged (sum of stages)",
               color=[col[m] for m in methods])
    axs[2].set_xticks(xm)
    axs[2].set_xticklabels(methods, rotation=20, fontsize=7)
    axs[2].set_title("training time (min)", fontsize=9)
    axs[2].legend(fontsize=6)
    bar(axs[3], [[_peak(r) for r in fin[m]] for m in methods], "peak GPU memory (GB)", "{:.2f}")
    plt.tight_layout()
    plt.savefig(f"{out}/method_comparison.png", dpi=150)
    plt.close()
    print(f"saved timing/loss plots in {out}/")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default=".")
    ap.add_argument("--conv_frac", type=float, default=0.9,
                    help="a stage counts as converged when the smoothed training loss covered this fraction of its drop")
    args = ap.parse_args()

    groups, meta, order, base = defaultdict(list), defaultdict(list), None, None
    for path in sorted(glob.glob(f"{args.root}/results/*/results.json")):
        d = json.load(open(path))
        if order is None:
            order, base = d["order"], d["R"]["base"]
        if d["order"] != order or d["order"][-1] not in d["R"]:
            print(f"skipping {path} (different task order or unfinished run)")
            continue
        method = re.sub(r"_s\d+$", "", os.path.basename(os.path.dirname(path)))
        groups[method].append(d["R"])
        meta[method].append(d)
    if not groups:
        print("no finished runs found")
        return

    rows = []
    print(f"\nTask order: {' -> '.join(order)}   (zero-shot: " +
          ", ".join(f"{t} {base[t]:.3f}" for t in order) + ")\n")
    header = ["method", "seeds", "avg_final", "forgetting", "BWT"] + [f"final_{t}" for t in order]
    print("| " + " | ".join(header) + " |")
    print("|" + "---|" * len(header))
    for m, runs in groups.items():
        mt = np.array([metrics(R, order) for R in runs])
        finals = {t: [R[order[-1]][t] for R in runs] for t in order}
        row = [m, str(len(runs)), ms(mt[:, 0]), ms(mt[:, 1]), ms(mt[:, 2])] + [ms(finals[t]) for t in order]
        rows.append(row)
        print("| " + " | ".join(row) + " |")

    os.makedirs(f"{args.root}/results", exist_ok=True)
    with open(f"{args.root}/results/summary.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)

    th, tr = timing_table(meta, order, args.conv_frac)
    if tr:
        print("\nTiming (minutes, mean ± std over seeds)\n")
        print("| " + " | ".join(th) + " |")
        print("|" + "---|" * len(th))
        for r_ in tr:
            print("| " + " | ".join(r_) + " |")
        with open(f"{args.root}/results/summary_timing.csv", "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(th)
            w.writerows(tr)
        sh, sr = stage_table(meta, order, args.conv_frac)
        print("\nPer stage (mean ± std over seeds)\n")
        print("| " + " | ".join(sh) + " |")
        print("|" + "---|" * len(sh))
        for r_ in sr:
            print("| " + " | ".join(r_) + " |")
        with open(f"{args.root}/results/summary_stages.csv", "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(sh)
            w.writerows(sr)
    training_plots(meta, order, args.root, args.conv_frac)

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    methods = list(groups)
    width = 0.8 / (len(methods) + 1)
    x = np.arange(len(order))
    plt.figure(figsize=(8, 4))
    plt.bar(x - 0.4 + width / 2, [base[t] for t in order], width, label="zero-shot", color="lightgray")
    for i, m in enumerate(methods):
        means = [np.mean([R[order[-1]][t] for R in groups[m]]) for t in order]
        errs = [np.std([R[order[-1]][t] for R in groups[m]]) for t in order]
        plt.bar(x - 0.4 + width * (i + 1.5), means, width, yerr=errs, label=m)
    plt.xticks(x, order)
    plt.ylabel("score after the final stage")
    plt.title("Performance on each task after training on all tasks")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{args.root}/results/final_scores.png", dpi=150)
    print(f"\nsaved {args.root}/results/summary.csv and final_scores.png")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile requirements.txt
torch
transformers
peft
accelerate
# WikiSQL on the Hub is script-based, which needs datasets<4 (and trust_remote_code)
datasets>=2.19,<4.0
scikit-learn
rouge_score
matplotlib
numpy
huggingface_hub

## 2b. Install requirements
Pins `datasets<4`. **Then restart the session** (Runtime -> Restart session).
The cell also removes Colab's preinstalled `torchao`, which is too old for the current `peft` and makes `get_peft_model` fail.


In [ ]:
!pip install -q -r requirements.txt
# Colab ships an old torchao (0.10.0) that peft refuses to work with (needs >= 0.16). This project does not use it.
!pip uninstall -y -q torchao

### Restart the session now, then continue below.

## 3. Settings + setup (the only cell you edit; re-run after every restart/reconnect)

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/s2l_final'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

# ----------------------------- EDIT THESE -----------------------------
S2L_REPO = "YOUR_HF_USERNAME/s2l-exchange"    # DATASET repo from the 4 machine notebooks (data/, buffers/, baseline results)
HF_REPO  = S2L_REPO                           # hand-off goes into the SAME repo ...
HF_FOLDER = "s2l_handoff"                     # ... in this new folder (never touches data/, buffers/, results/)
RATIOS = [0.02, 0.05]                         # S2L buffer ratios to run and compare (buffers/s2l_<ratio>/ must exist in S2L_REPO)
SEED  = 0
MICRO_BS = 2                                  # sequences per GPU step (effective batch stays 16). T4: 2. Bigger GPU (L4/A100): 4 or 8 is faster
# -----------------------------------------------------------------------
assert "YOUR_HF_USERNAME" not in S2L_REPO, "Edit S2L_REPO first"

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (write access): ")
os.environ["S2L_REPO"] = S2L_REPO            # used by sync.py
os.environ["HF_FOLDER"] = HF_FOLDER          # used by s2l_lib.py (hand-off folder)
os.environ["S2L_MICRO_BS"] = str(MICRO_BS)       # used by s2l_lib.py
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less memory fragmentation
from huggingface_hub import HfApi
print("Hugging Face user:", HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"])
print("repo (dataset):", S2L_REPO, "| hand-off folder:", HF_FOLDER + "/", "| ratios", RATIOS, "| seed", SEED)
import datasets; print("datasets", datasets.__version__, "(should be < 4.0)")
RUNS = [f"s2l_{r}_s{SEED}" for r in RATIOS]
!ls

# PART A: pull data + buffers, train stages 1-2, push to Hugging Face

Stage 1 trains on GSM8K; stage 2 on WikiSQL + the S2L buffer of GSM8K. The model is evaluated on **all** tasks (score and held-out loss) after each stage, with timing and training-loss logs. The script checks the buffers and your HF write access **before** training.

## A1. Pull the cached data and S2L buffers from the exchange repo
Blocks until the files exist in the repo, then downloads them. The buffers store **indices** into `data/*.json`, so both must come from the same repo.

In [ ]:
need = " ".join([f"data/{t}.json" for t in ["gsm8k", "wikisql", "xsum", "pubmedqa"]] +
                [f"buffers/s2l_{r}/{t}.json" for r in RATIOS for t in ["gsm8k", "wikisql", "xsum"]])
!python sync.py wait {need} --timeout 1
!python sync.py pull data buffers
!ls data buffers

In [ ]:
import json
for t, n_tr in [("gsm8k", 7000), ("wikisql", 8000), ("xsum", 8000), ("pubmedqa", 700)]:
    d = json.load(open(f"data/{t}.json"))
    print(f"{t:9s} {len(d['train'])} train / {len(d['test'])} test", "" if len(d["train"]) == n_tr else f"  <-- expected {n_tr} train")
for r in RATIOS:
    for t in ["gsm8k", "wikisql", "xsum"]:
        idx = json.load(open(f"buffers/s2l_{r}/{t}.json"))
        print(f"ratio {r}: buffer {t:8s} {len(idx)} rows (max index {max(idx)})")

## A2. Stages 1-2 and upload the hand-off

In [ ]:
for r in RATIOS:
    print(f"\n################ stages 1-2, ratio {r} ################")
    !python s2l_stage12.py --hf_repo {HF_REPO} --ratio {r} --seed {SEED}

In [ ]:
# What is in the hand-off folder now
for f in HfApi(token=os.environ["HF_TOKEN"]).list_repo_files(HF_REPO, repo_type="dataset"):
    if f.startswith(HF_FOLDER + "/"):
        print(f)

**If training finished but the upload failed** (network error at the very end), do not retrain: results are in Drive. Run
`!python s2l_stage12.py --hf_repo {HF_REPO} --ratio 0.05 --seed {SEED} --push_only` (use the ratio that failed)

**Tell person B** (if someone else runs Part B): `S2L_REPO`, `HF_FOLDER`, `RATIOS`, `SEED`, and to accept the collaborator invite.

# PART B: pull the hand-off, train stages 3-4, push the final results

Needs sections 1-3 (same `S2L_REPO`, `HF_FOLDER`, `RATIOS`, `SEED`) and access to the repo. It does **not** need `data/` or `buffers/`: they come with the hand-off.

## B1. Stages 3-4
Downloads the hand-off, **verifies** it (same `common.py`, SHA-256 of every file, task order, ratio), reads epochs / lr / seed from the stages 1-2 run, continues from the stage-2 adapter (XSum + GSM8K/WikiSQL buffers, then PubMedQA + the three buffers), evaluates on all tasks after each stage, and uploads the final adapters + merged `results.json`.

If it stops with *Hand-off check FAILED*, the message names the file that differs.

In [ ]:
for r in RATIOS:
    print(f"\n################ stages 3-4, ratio {r} ################")
    !python s2l_stage34.py --hf_repo {HF_REPO} --ratio {r} --seed {SEED}

If the final upload failed after training, do not retrain: run the same command for that ratio with `--push_only` added.

# PART C: aggregate everything and plot

Pulls the finished **baseline** runs (`none`, `random`, `full`) from the exchange repo and puts them next to the S2L run in one table.
The S2L run is deliberately **not** pulled from the exchange repo: a run with the same name from the 4-machine setup would overwrite this one.

## C1. Collect the runs

In [ ]:
api = HfApi(token=os.environ["HF_TOKEN"])
have = set(api.list_repo_files(S2L_REPO, repo_type="dataset"))
baselines = [f"none_s{SEED}", f"full_s{SEED}"] + [f"random_{r}_s{SEED}" for r in RATIOS]
found = [b for b in baselines if f"results/{b}/results.json" in have]
print("baselines found in exchange repo:", found)
print("baselines missing (not finished/pushed, or never run):", [b for b in baselines if b not in found])
baseline_paths = " ".join(f"results/{b}" for b in found)

In [ ]:
!python sync.py pull {baseline_paths}

In [ ]:
# The S2L runs: use the local copy if it is complete, otherwise fetch it from the hand-off folder
import json
order = ["gsm8k", "wikisql", "xsum", "pubmedqa"]
def complete(run):
    p = f"results/{run}/results.json"
    return os.path.exists(p) and all(t in json.load(open(p))["R"] for t in order)
for r, run in zip(RATIOS, RUNS):
    if not complete(run):
        get_ipython().system(f"python s2l_lib.py --hf_repo {HF_REPO} --ratio {r} --seed {SEED}")
    print(run, "complete locally:", complete(run))
!ls results

## C2. Tables and score plot

In [ ]:
!python aggregate_results.py
import pandas as pd
from IPython.display import Image, display
print("Quality: avg final score, forgetting, BWT, final score per task")
display(pd.read_csv("results/summary.csv"))
if os.path.exists("results/summary_timing.csv"):
    print("Totals per run: training time, time to converge, held-out loss, peak GPU memory (runs with logs only)")
    display(pd.read_csv("results/summary_timing.csv"))
    print("Per stage")
    display(pd.read_csv("results/summary_stages.csv"))
display(Image("results/final_scores.png"))

## C3. Loss, convergence-time, memory and forgetting plots
`method_comparison` puts the headline numbers side by side; the others give the per-stage detail.

In [ ]:
for p in ["method_comparison", "convergence_time", "peak_memory", "time_per_stage", "train_loss_by_stage",
          "train_loss_vs_time", "test_loss_forgetting", "time_vs_quality"]:
    f = f"results/{p}.png"
    if os.path.exists(f):
        print(p); display(Image(f))

## C4. (Optional) Save the tables and plots to the Hugging Face repo (hand-off folder)

In [ ]:
HfApi(token=os.environ["HF_TOKEN"]).upload_folder(
    folder_path="results", path_in_repo=f"{HF_FOLDER}/summary", repo_id=HF_REPO, repo_type="dataset",
    allow_patterns=["*.png", "*.csv"], commit_message="aggregate tables and plots")
print(f"uploaded results/*.png and *.csv to {HF_REPO} under {HF_FOLDER}/summary/")

## Notes
- **Timing, loss and memory plots** (`method_comparison`, `convergence_time`, `peak_memory`, `time_per_stage`, `train_loss_*`, `test_loss_forgetting`, `time_vs_quality`) only contain runs whose `results.json` has timing and held-out-loss logs. The S2L runs from this notebook have them; baselines made by the older `sequential_finetune.py` do not, so they appear in the score table and `final_scores.png` but not in those plots.
- **Time to converge** is measured on the training loss: the first logged point (every 20 steps) where the 3-point smoothed loss has covered 90% of its drop in that stage. Change it with `!python aggregate_results.py --conv_frac 0.95`. It is minutes since the stage started; `summary_stages.csv` also gives the step.
- **Peak memory** is torch's peak allocated GPU memory during each stage's training (evaluation not included). The S2L ratio changes how many rows are trained, not the batch size, so expect memory to differ much less than time does.
- **Baselines for 5%:** the 4-machine setup only ran `random` at 2%, so there is no `random_0.05` unless you run it.
- **Other seed:** change `SEED` in section 3. Runs are named by ratio and seed, so nothing is overwritten.
- A stage that is cut off (disconnect) must be re-run from its start; keep the tab open during long steps.- **CUDA out of memory:** lower `MICRO_BS` in section 3 (2, then 1). The effective batch stays 16 (gradient accumulation), so the training is equivalent; only speed and peak memory change. Use the same `MICRO_BS` for both ratios so their time and memory are comparable. If a run crashed, re-run that stage command: a stage that did not finish restarts from its beginning.
